# training-step-cycle — worked example 3: Full epoch loop with the 5-call cycle over a DataLoader

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `training-step-cycle`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A training epoch iterates the training DataLoader once, running the 5-call cycle for each mini-batch. The key difference from a single-batch example is that `zero_grad` must be called each iteration so that gradients from one mini-batch don't pollute the next. After a full epoch, the model has seen every training example once and the optimizer has taken one update per batch.

## Worked solution

**Step 1 – Create a DataLoader.** We wrap `(X, Y)` in a `TensorDataset` and pass it to `DataLoader` with a batch size. Each iteration yields one mini-batch `(xb, yb)`.

**Step 2 – Epoch loop.** We call `model.train()` at the start of each epoch (good habit; some modules like Dropout and BN need it). Then we iterate the DataLoader.

**Step 3 – Per-batch 5-call cycle.** For each `(xb, yb)`: `pred = model(xb)`, `loss = loss_fn(pred, yb)`, snapshot, `loss.backward()`, `optimizer.step()`, `optimizer.zero_grad()`.

**Step 4 – Epoch summary.** After the loop, we compute the mean loss over all batches in the epoch. Watching this per-epoch mean decrease across epochs is the primary training signal.

In [ ]:
import torch as t
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

def worked3_full_epoch(n_epochs=5):
    """
    Full training loop over a DataLoader.
    Returns per-epoch average loss list.
    """
    t.manual_seed(99)
    # Dataset: y = 2x1 - x2
    X = t.randn(80, 2)
    Y = (2 * X[:, 0] - X[:, 1]).unsqueeze(1)
    loader = DataLoader(TensorDataset(X, Y), batch_size=16, shuffle=False)

    model = nn.Linear(2, 1)
    optimizer = t.optim.SGD(model.parameters(), lr=0.1)
    loss_fn = nn.MSELoss()

    epoch_losses = []
    for epoch in range(n_epochs):
        model.train()
        batch_losses = []
        for xb, yb in loader:
            pred = model(xb)             # forward
            loss = loss_fn(pred, yb)     # loss
            batch_losses.append(loss.item())
            loss.backward()              # backward
            optimizer.step()             # step
            optimizer.zero_grad()        # zero_grad
        epoch_losses.append(sum(batch_losses) / len(batch_losses))

    return epoch_losses

epoch_losses = worked3_full_epoch()
for i, v in enumerate(epoch_losses):
    print(f'epoch {i}: loss = {v:.4f}')